In [1]:
# !pip install -U openai

In [2]:
# from google.colab import userdata
import os
from dotenv import load_dotenv

# .env 파일 로드
load_dotenv()

# 환경 변수에서 가져오기
api_key = os.getenv("OPENAI_API_KEY")

# Colab에 저장한 Secret에서 API 키 가져와 환경 변수에 등록
os.environ["OPENAI_API_KEY"] = api_key

# 제대로 설정됐는지 확인 (키 값은 출력하지 않도록 주의)
print("API Key configured:", "OPENAI_API_KEY" in os.environ)

API Key configured: True


In [3]:
from openai import OpenAI

# OpenAI 클라이언트 인스턴스 생성
client = OpenAI()

##실습 5.1. Chain of Thought 기법 실습

In [4]:
# Chain-of-Thought 없이 답변 생성
question = "철수는 사과 10개를 가지고 있었는데, 그 중 3개를 먹은 후 5개를 더 구입했습니다. 철수가 현재 가지고 있는 사과는 몇 개일까요?"
prompt1 = question  # 별도 지시 없이 바로 질문만 던집니다.
response1 = client.chat.completions.create(
    messages=[
        {"role": "user", "content": prompt1}
    ],
    model="gpt-4o-mini",
    temperature=0
)
print("CoT 미적용 답변:", response1.choices[0].message.content)

CoT 미적용 답변: 철수는 처음에 사과 10개를 가지고 있었습니다. 그 중 3개를 먹었으므로 남은 사과는 10 - 3 = 7개입니다. 이후 5개를 더 구입했으므로 현재 가지고 있는 사과는 7 + 5 = 12개입니다. 

따라서 철수가 현재 가지고 있는 사과는 12개입니다.


In [5]:
# Chain-of-Thought 프롬프트와 함께 답변 생성
cot_prompt = "(생각을 단계별로 진행합니다)\n" + question  # 단계적 사고를 지시하는 문구를 추가
response2 = client.chat.completions.create(
    messages=[
        {"role": "user", "content": cot_prompt}
    ],
    model="gpt-4o-mini",
    temperature=0
)
print("CoT 적용 답변:", response2.choices[0].message.content)

CoT 적용 답변: 철수가 처음에 가지고 있던 사과의 개수는 10개입니다. 

1. 철수가 3개를 먹었습니다:  
   10개 - 3개 = 7개

2. 그 후 5개를 더 구입했습니다:  
   7개 + 5개 = 12개

따라서, 철수가 현재 가지고 있는 사과는 12개입니다.


##5.2. Self-Consistency 기법 실습

In [6]:
import collections

question = "어떤 수를 3배한 결과에 4를 더하면 19가 됩니다. 그 수는 무엇일까요? 단계별로 풀어보세요."
messages = [{"role": "user", "content": question}]
answers = []

# 동일한 질문에 대해 여러 번 응답 생성 (예: 5회)
for i in range(5):
    response = client.chat.completions.create(
        messages=messages,
        model="gpt-4o-mini",
        temperature=1.0
    )
    answer = response.choices[0].message.content.strip()
    answers.append(answer)
    print(f"응답{i+1}: {answer}")

# 최종 답 도출 – 모든 응답의 마지막 줄(결과)을 모아 다수결 투표
final_answers = [ans.split()[-1] for ans in answers]  # 각 답변에서 마지막 단어를 가져와서 (여기서는 '5'같은 숫자일 것이라 가정)
counter = collections.Counter(final_answers)
final_answer = counter.most_common(1)[0][0]  # 가장 빈도 높은 답
print("\n다수결에 따른 최종 답:", final_answer)

응답1: 주어진 문제를 해결하기 위해, 어떤 수를 \( x \)라고 가정합시다. 그러면 문제에서 주어진 내용을 수식으로 표현할 수 있습니다.

1. 어떤 수를 3배한 결과: \( 3x \)
2. 여기에 4를 더하면: \( 3x + 4 \)
3. 이 결과는 19와 같다: \( 3x + 4 = 19 \)

위의 수식을 사용하여 \( x \)를 구해보겠습니다.

### 1단계: 수식 정리
\[ 
3x + 4 = 19 
\]

### 2단계: 양변에서 4를 뺍니다.
\[
3x + 4 - 4 = 19 - 4 
\]
\[
3x = 15 
\]

### 3단계: 양변을 3으로 나눕니다.
\[
\frac{3x}{3} = \frac{15}{3} 
\]
\[
x = 5 
\]

따라서, 구하고자 하는 수는 **5**입니다.

### 검증
이제 구한 답이 맞는지 확인해 보겠습니다.

1. 5를 3배하면: \( 3 \times 5 = 15 \)
2. 여기에 4를 더하면: \( 15 + 4 = 19 \)

결과가 맞으므로, 결론은 **5**입니다.
응답2: 주어진 문제를 단계별로 풀어보겠습니다.

1. **문제를 이해하기:** 어떤 수를 'x'라고 하겠습니다. 이 수를 3배한 결과에 4를 더하면 19가 된다는 조건이 있습니다. 따라서 우리는 다음과 같은 수식을 세울 수 있습니다.

   \[
   3x + 4 = 19
   \]

2. **방정식 정리하기:** 이제 이 방정식을 풀어보겠습니다. 첫 번째 단계로, 양쪽에서 4를 빼줘야 합니다.

   \[
   3x + 4 - 4 = 19 - 4
   \]

   이 식을 정리하면:

   \[
   3x = 15
   \]

3. **x 구하기:** 이제 양쪽을 3으로 나누어 'x'를 구합니다.

   \[
   x = \frac{15}{3}
   \]

   계산을 하면:

   \[
   x = 5
   \]

4. **결과 확인하기:** 마지막으로, 우리가 찾은 'x' 값이 문제의 조건을 충족하는지 확인합니다. 'x'가 5일 때

##API를 이용한 Iteration : 1번의 요청에 5개의 응답

In [7]:
import collections

question = "어떤 수를 3배한 결과에 4를 더하면 19가 됩니다. 그 수는 무엇일까요? 단계별로 풀어보세요."
messages = [{"role": "user", "content": question}]
answers = []

# 동일한 질문에 대해 여러 번 응답 생성 (예: 5회)
response = client.chat.completions.create(
    messages=messages,
    model="gpt-4o-mini",
    temperature=1.0,
    n=5
)
for i in range(5):
    answer = response.choices[i].message.content.strip()
    answers.append(answer)
    print(f"응답{i+1}: {answer}")

# 최종 답 도출 – 모든 응답의 마지막 줄(결과)을 모아 다수결 투표
final_answers = [ans.split()[-1] for ans in answers]  # 각 답변에서 마지막 단어를 가져와서 (여기서는 '5'같은 숫자일 것이라 가정)
counter = collections.Counter(final_answers)
final_answer = counter.most_common(1)[0][0]  # 가장 빈도 높은 답
print("\n다수결에 따른 최종 답:", final_answer)

응답1: 어떤 수를 \( x \)라고 가정하겠습니다. 문제에서 주어진 조건을 수식으로 나타내면 다음과 같습니다.

\[
3x + 4 = 19
\]

이제 이 식을 풀어보겠습니다.

### 1단계: 4를 양변에서 빼기
식의 양변에서 4를 빼줍니다.

\[
3x + 4 - 4 = 19 - 4
\]

이렇게 되면,

\[
3x = 15
\]

### 2단계: 양변을 3으로 나누기
다음으로, \( x \)를 구하기 위해 양변을 3으로 나눕니다.

\[
\frac{3x}{3} = \frac{15}{3}
\]

이렇게 되면,

\[
x = 5
\]

따라서, 어떤 수는 \( 5 \)입니다.

### 검증
이제 결과를 검증해 보겠습니다. \( x = 5 \)일 때,

\[
3 \times 5 + 4 = 15 + 4 = 19
\]

이므로 조건에 맞습니다. 따라서, 정답은 

\[
\boxed{5}
\]

입니다.
응답2: 문제를 단계별로 풀어보겠습니다.

1. **문제 분석**: '어떤 수'를 x라고 합시다. 그러면 주어진 조건을 수식으로 표현할 수 있습니다. 수를 3배한 결과에 4를 더하면 19가 된다는 것은 다음과 같은 식이 성립합니다.

   \[
   3x + 4 = 19
   \]

2. **식 정리**: 이제 이 식을 풀어봅시다. 우선 양쪽에서 4를 빼겠습니다.

   \[
   3x + 4 - 4 = 19 - 4
   \]

   이렇게 하면 다음과 같은 식이 됩니다.

   \[
   3x = 15
   \]

3. **x 구하기**: 이제 x를 구하기 위해 양쪽을 3으로 나눕니다.

   \[
   x = \frac{15}{3}
   \]

   계산하면:

   \[
   x = 5
   \]

4. **결과 확인**: 마지막으로 구한 x의 값을 원래 조건에 대입하여 확인해보겠습니다.

   \[
   3 \times 5 + 4 = 15 + 4 = 19
   \]

   조건이 맞으므로, 계산이 올바릅니다.

따라서, '어떤 수'는 **5**입니다.
응답

##실습5.3. Reflexion 기법 산술문제 실습

In [8]:
# 1. 초기 질문에 대한 1차 답변 생성
question = "7의 2승에 5를 곱한 값은 무엇인가?"
response1 = client.chat.completions.create(
    messages=[
        {"role": "user", "content": question}
    ],
    model="gpt-4o-mini",
    temperature=0
)
answer1 = response1.choices[0].message.content.strip()
print("1차 답변:", answer1)

# 2. 1차 답변에 대한 피드백 생성 (모델 스스로에게 해볼 수도 있지만, 여기서는 정답 알고 있다고 가정하고 직접 피드백 작성)
correct_answer = 245
if str(correct_answer) in answer1:
    feedback = "정답입니다. 잘 해결했어요!"
    need_retry = False
else:
    feedback = "오답입니다. 계산을 다시 해보세요. (힌트: 7의 제곱값을 정확히 구한 후 곱하세요.)"
    need_retry = True

print("피드백:", feedback)

1차 답변: 7의 2승은 \( 7^2 = 49 \)입니다. 여기에 5를 곱하면:

\[ 49 \times 5 = 245 \]

따라서, 7의 2승에 5를 곱한 값은 245입니다.
피드백: 정답입니다. 잘 해결했어요!


In [9]:
# 3. 피드백을 포함한 새로운 프롬프트 구성하여 2차 답변 생성 (만약 재시도가 필요한 경우)
if need_retry:
    retry_prompt = f"이전 답변: {answer1}\n피드백: {feedback}\n따라서 답을 다시 구해보세요."
    response2 = client.chat.completions.create(
        messages=[
            {"role": "user", "content": retry_prompt}
        ],
        model="gpt-4o-mini",
        temperature=0
    )
    answer2 = response2.choices[0].message.content.strip()
    print("2차 답변:", answer2)

##실습5.4. Reflexion 기법 에세이작성 실습

In [10]:
# 에세이 작성 1차 시도
prompt = "기후 변화의 원인과 해결 방안에 대해 간략한 에세이를 작성하시오."
response_essay1 = client.chat.completions.create(
    messages=[
        {"role": "user", "content": prompt}
    ],
    model="gpt-4o-mini",
    temperature=0
)
essay1 = response_essay1.choices[0].message.content
print("1차 에세이:\n", essay1)

1차 에세이:
 기후 변화는 현대 사회가 직면한 가장 심각한 환경 문제 중 하나로, 지구의 평균 기온 상승, 해수면 상승, 극단적인 기상 현상 등의 형태로 나타나고 있다. 이러한 기후 변화의 주요 원인은 인간 활동에 의해 발생하는 온실가스의 배출이다. 특히, 화석 연료의 연소, 산업 활동, 농업 및 임업 등에서 발생하는 이산화탄소(CO2), 메탄(CH4), 아산화질소(N2O) 등의 온실가스가 대기 중에 축적되어 지구의 온도를 상승시키고 있다.

기후 변화의 해결 방안은 여러 가지가 있지만, 가장 효과적인 방법은 온실가스 배출을 줄이는 것이다. 이를 위해서는 재생 가능 에너지의 사용을 확대해야 한다. 태양광, 풍력, 수력 등 청정 에너지원으로의 전환은 화석 연료 의존도를 줄이고, 지속 가능한 에너지 시스템을 구축하는 데 기여할 수 있다. 또한, 에너지 효율성을 높이는 기술 개발과 보급도 중요하다. 건물, 교통수단, 산업 공정에서 에너지를 절약하는 방법을 도입함으로써 온실가스 배출을 줄일 수 있다.

또한, 개인과 기업의 인식 변화도 필요하다. 소비자들이 친환경 제품을 선택하고, 기업들이 지속 가능한 경영 방침을 채택하도록 유도하는 것이 중요하다. 정부는 이러한 변화를 촉진하기 위해 정책적 지원과 인센티브를 제공해야 하며, 국제 사회와의 협력을 통해 기후 변화 대응을 위한 글로벌 노력을 강화해야 한다.

결론적으로, 기후 변화는 인류의 생존과 미래에 중대한 영향을 미치는 문제로, 이를 해결하기 위해서는 온실가스 배출을 줄이고, 지속 가능한 에너지 시스템을 구축하며, 사회 전반의 인식 변화를 이끌어내는 노력이 필요하다. 이러한 노력이 모여 기후 변화에 대한 효과적인 대응이 이루어질 수 있을 것이다.


In [11]:
# 피드백 작성 (여기서는 사람이 수동으로 평가하여 문자열 작성한다고 가정)
feedback = (
    "피드백:\n"
    " - 서론에서 주제 소개가 부족합니다.\n"
    " - 원인에 대한 설명이 모호하며 구체적 예시와 근거를 제시할 통계자료가 없습니다.\n"
    " - 해결 방안을 최소 10가지 이상 제시해주세요.\n"
    "위 사항을 반영하여 에세이를 수정해주세요."
)
# 개선 프롬프트 구성
retry_prompt = essay1 + "\n\n" + feedback
response_essay2 = client.chat.completions.create(
    messages=[
        {"role": "user", "content": retry_prompt}
    ],
    model="gpt-4o-mini",
    temperature=0
)
essay2 = response_essay2.choices[0].message.content
print("2차 에세이:\n", essay2)

2차 에세이:
 기후 변화는 현대 사회가 직면한 가장 심각한 환경 문제 중 하나로, 이는 지구의 평균 기온 상승, 해수면 상승, 극단적인 기상 현상 등의 형태로 나타나고 있다. 이러한 변화는 인류의 생존과 미래에 중대한 영향을 미치며, 이를 해결하기 위한 긴급한 노력이 필요하다. 기후 변화의 주요 원인은 인간 활동에 의해 발생하는 온실가스의 배출로, 특히 화석 연료의 연소, 산업 활동, 농업 및 임업 등에서 발생하는 이산화탄소(CO2), 메탄(CH4), 아산화질소(N2O) 등의 온실가스가 대기 중에 축적되어 지구의 온도를 상승시키고 있다. 예를 들어, 2020년 기준으로 전 세계 이산화탄소 배출량은 약 33억 톤에 달하며, 이는 산업화 이후 급격히 증가한 수치이다.

기후 변화의 해결 방안은 여러 가지가 있으며, 가장 효과적인 방법은 온실가스 배출을 줄이는 것이다. 이를 위해 다음과 같은 10가지 이상의 구체적인 방안을 제시할 수 있다:

1. **재생 가능 에너지 확대**: 태양광, 풍력, 수력 등 청정 에너지원으로의 전환을 통해 화석 연료 의존도를 줄인다.
2. **에너지 효율성 향상**: 건물, 교통수단, 산업 공정에서 에너지를 절약하는 기술을 도입하여 에너지 소비를 줄인다.
3. **전기차 및 대중교통 활성화**: 전기차 사용을 장려하고 대중교통 시스템을 개선하여 교통 부문에서의 온실가스 배출을 줄인다.
4. **산업 공정 개선**: 산업에서 발생하는 온실가스를 줄이기 위한 청정 기술 및 공정 개선을 도입한다.
5. **농업의 지속 가능성 증대**: 지속 가능한 농업 방법을 채택하고, 비료 및 농약 사용을 줄여 메탄과 아산화질소 배출을 감소시킨다.
6. **임업 및 산림 보호**: 산림을 보호하고 재조림 프로젝트를 통해 이산화탄소를 흡수하는 자연적인 방법을 활용한다.
7. **폐기물 관리 개선**: 쓰레기 매립지에서 발생하는 메탄을 줄이기 위해 재활용 및 퇴비화 프로그램을 강화한다.
8. **친환경 제품 소비 촉진**: 소비자들이 친환경 제품을 선택하도

In [12]:
# 피드백 작성 (여기서는 사람이 수동으로 평가하여 문자열 작성한다고 가정)
feedback_2 = (
    "피드백:\n"
    " - 관련 사실을 실증할만한 근거의 통계 자료가 없습니다.\n"
    "위 사항을 반영하여 에세이를 수정해주세요."
)
# 개선 프롬프트 구성
retry_prompt_2 = essay1 + "\n\n" + feedback_2
response_essay3 = client.chat.completions.create(
    messages=[
        {"role": "user", "content": retry_prompt_2}
    ],
    model="gpt-4o-mini",
    temperature=0
)
essay3 = response_essay3.choices[0].message.content
print("3차 에세이:\n", essay3)

3차 에세이:
 기후 변화는 현대 사회가 직면한 가장 심각한 환경 문제 중 하나로, 지구의 평균 기온 상승, 해수면 상승, 극단적인 기상 현상 등의 형태로 나타나고 있다. 2021년 IPCC(기후 변화에 관한 정부 간 패널) 보고서에 따르면, 1850년 이후 지구의 평균 기온은 약 1.1도 상승했으며, 이는 주로 인간 활동에 의해 발생하는 온실가스의 배출에 기인하고 있다. 특히, 화석 연료의 연소, 산업 활동, 농업 및 임업 등에서 발생하는 이산화탄소(CO2), 메탄(CH4), 아산화질소(N2O) 등의 온실가스가 대기 중에 축적되어 지구의 온도를 상승시키고 있다. 2020년에는 전 세계적으로 약 36.4억 톤의 CO2가 배출되었으며, 이는 2019년 대비 약 6% 감소했지만 여전히 기후 목표에 미치지 못하는 수치이다.

기후 변화의 해결 방안은 여러 가지가 있지만, 가장 효과적인 방법은 온실가스 배출을 줄이는 것이다. 이를 위해서는 재생 가능 에너지의 사용을 확대해야 한다. 2020년 기준으로 전 세계 전력의 약 29%가 재생 가능 에너지에서 생산되었으며, 이는 2010년 대비 두 배 이상 증가한 수치이다. 태양광, 풍력, 수력 등 청정 에너지원으로의 전환은 화석 연료 의존도를 줄이고, 지속 가능한 에너지 시스템을 구축하는 데 기여할 수 있다. 또한, 에너지 효율성을 높이는 기술 개발과 보급도 중요하다. 예를 들어, 에너지 효율적인 건물은 기존 건물보다 에너지를 최대 30%까지 절약할 수 있으며, 이는 온실가스 배출을 줄이는 데 큰 도움이 된다.

또한, 개인과 기업의 인식 변화도 필요하다. 2021년 조사에 따르면, 소비자의 70% 이상이 친환경 제품을 선호한다고 응답했으며, 이는 기업들이 지속 가능한 경영 방침을 채택하도록 유도하는 중요한 요소가 된다. 정부는 이러한 변화를 촉진하기 위해 정책적 지원과 인센티브를 제공해야 하며, 예를 들어, 재생 가능 에너지에 대한 세금 감면이나 보조금 지원을 통해 기업과 개인이 친환경 선택을 할 수 있도록 유도할 수 있다